# 02 - Train Distribution Function (RealNVP)

Train the normalizing flow (RealNVP) to model the distribution function of tracer particles.

**Advantage of Jupyter**: JIT-compiled functions stay cached in memory across cells. Re-running training cells is near-instant after the first compilation.

In [ ]:
import jax
print(f"JAX backend: {jax.default_backend()}")
print(f"JAX devices: {jax.devices()}")

In [ ]:
from dpjax.config import load_config, merge_config
from dpjax.paths import PROJECT_ROOT, DATA_DIR, RUNS_DIR

print(f"Project root: {PROJECT_ROOT}")

## 1. Load and Customize Configuration

Start from the standard YAML config and override parameters for a quick test run.

In [ ]:
cfg = load_config("configs/df_plummer.yaml")

# Override for a quick test (reduce epochs / batch size as needed)
cfg = merge_config(cfg, {
    "train": {
        "epochs": 4,          # quick test; use 256 for full training
        "batch_size": 4096,
        "log_every": 20,
        "ckpt_every": 100,
    }
})

print("Config:")
import yaml
print(yaml.safe_dump(cfg, sort_keys=False))

## 2. Train DF

In [ ]:
from experiments.train_df import run_df_training

DATA_PATH = DATA_DIR / "plummer_n131072.h5"
RUN_DIR   = RUNS_DIR / "plummer" / "df"

result = run_df_training(
    config=cfg,
    data_path=DATA_PATH,
    run_dir=RUN_DIR,
)

print(f"\nTraining complete. Final step: {result['final_step']}")

## 3. Inspect Training Metrics

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import csv

metrics_path = RUN_DIR / "metrics.csv"

with metrics_path.open() as f:
    reader = csv.DictReader(f)
    rows = [r for r in reader]

steps = np.array([float(r["step"]) for r in rows])
loss  = np.array([float(r["loss"]) for r in rows])
p99   = np.array([float(r["score_p99"]) for r in rows])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(steps, loss, lw=1.2)
ax1.set_xlabel("step"); ax1.set_ylabel("NLL loss")
ax1.set_title("Training Loss"); ax1.grid(True, alpha=0.2)

ax2.plot(steps, p99, lw=1.2, color="tab:orange")
ax2.set_xlabel("step"); ax2.set_ylabel("|score| p99")
ax2.set_title("Score p99"); ax2.grid(True, alpha=0.2)

fig.tight_layout()
plt.show()

## 4. Quick Sanity Check: Sample from Trained Flow

In [ ]:
import jax.numpy as jnp
from dpjax.flows.realnvp import RealNVP

model = result["model"]
params = result["params"]
normalizer = result["normalizer"]

rng = jax.random.key(42)
x_std = model.apply({"params": params}, rng, 50000, method=RealNVP.sample)
eta_sampled = np.asarray(normalizer.inverse(np.asarray(x_std)))

# Compare x-y projection
from dpjax.data import load_eta_h5
eta_data = load_eta_h5(DATA_PATH)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4.5))

ax1.hist2d(eta_data[:, 0], eta_data[:, 1], bins=100, cmap="viridis", density=True)
ax1.set_xlabel("x"); ax1.set_ylabel("y")
ax1.set_title("Training Data"); ax1.set_aspect("equal")

ax2.hist2d(eta_sampled[:, 0], eta_sampled[:, 1], bins=100, cmap="viridis", density=True)
ax2.set_xlabel("x"); ax2.set_ylabel("y")
ax2.set_title("Flow Samples"); ax2.set_aspect("equal")

fig.suptitle("(x, y) Projection: Data vs Flow", fontsize=13)
fig.tight_layout()
plt.show()